# 13 — Partitioning Strategy Pattern

Purpose: control data layout for performance (reads/writes).

Process:

DATA
  |
  v
PARTITION BY COLUMN
  |
  v
OPTIMIZED STORAGE

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("partitioning-pattern")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

## Step 1 — Input

In [2]:
df = spark.createDataFrame(
    [
        ("e1","u1",10.0,"2026-01-01"),
        ("e2","u2",20.0,"2026-01-01"),
        ("e3","u1",30.0,"2026-01-02"),
        ("e4","u2",40.0,"2026-01-02"),
    ],
    ["event_id","user_id","amount","event_date"]
)

df.show()

+--------+-------+------+----------+
|event_id|user_id|amount|event_date|
+--------+-------+------+----------+
|      e1|     u1|  10.0|2026-01-01|
|      e2|     u2|  20.0|2026-01-01|
|      e3|     u1|  30.0|2026-01-02|
|      e4|     u2|  40.0|2026-01-02|
+--------+-------+------+----------+



## Step 2 — Repartition

In [3]:
df_repartitioned = df.repartition("event_date")
print(df_repartitioned.rdd.getNumPartitions())

1


## Step 3 — Write partitioned (simulated)

In [4]:
# Example (disabled for notebook safety):
# df.write.partitionBy("event_date").parquet("/data/events")
print("Partition by event_date recommended")

Partition by event_date recommended


## Step 4 — Skew example

In [5]:
skew_df = spark.createDataFrame(
    [("e"+str(i), "u1", i) for i in range(100)],
    ["event_id","user_id","amount"]
)

skew_df.groupBy("user_id").count().show()

+-------+-----+
|user_id|count|
+-------+-----+
|     u1|  100|
+-------+-----+



## Step 5 — Control

In [6]:
print("rows:", df.count())

rows: 4


In [7]:
spark.stop()